In [1]:
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException 
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import re
import time

In [2]:
def set_up_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless") 
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Tắt tính năng phát hiện tự động của trình duyệt
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) # Loại bỏ thông báo "Chrome is being controlled by automated test software"
    chrome_options.add_experimental_option('useAutomationExtension', False) # Vô hiệu hóa tiện ích mở rộng tự động hóa
    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [3]:
pharmacity_df = pd.read_csv("pharmacity_medicines_data_demo(4).csv")
pharmacity_df.shape[0]
pharmacity_df.tail()

,name,price,images,nha_san_xuat,noi_san_xuat,thuoc_can_ke_don,hoat_chat,chi_dinh_tom_tat,dang_bao_che,quy_cach,...,huong_dan_su_dung,cach_su_dung,than_trong,thong_tin_san_xuat,url,category,drug_type,category_link,category_name,error
2195,"Viên nén Gensler 5mg điều trị tăng huyết áp, s...",NaN,['https://prod-cdn.pharmacity.io/e-com/images/...,Davipharm,NaN,NaN,Ramipril,"Điều trị tăng huyết áp, suy tim",Viên nén,10 vỉ x 10 viên,...,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/gensler-5mg-hop-10-v...,"Thuốc tim mạch, huyết áp",Thuốc kê đơn,https://www.pharmacity.vn/rx-thuoc-tim-mach-hu...,"Thuốc tim mạch, huyết áp",NaN
2196,Viên nén Fenostad 160mg điều trị rối loạn mỡ m...,NaN,['https://prod-cdn.pharmacity.io/e-com/images/...,STELLA,NaN,NaN,Fenofibrat,Điều trị rối loạn mỡ máu,Viên nén,3 vỉ x 10 viên,...,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/fenostad-160-hop-3-v...,"Thuốc tim mạch, huyết áp",Thuốc kê đơn,https://www.pharmacity.vn/rx-thuoc-tim-mach-hu...,"Thuốc tim mạch, huyết áp",NaN
2197,Viên nén Irbesartan Domesco 150mg điều trị tăn...,NaN,['https://prod-cdn.pharmacity.io/e-com/images/...,Domesco,NaN,NaN,Irbesartan,Điều trị tăng huyết áp,Viên nén,2 vỉ x 14 viên,...,"Tác dụng phụ\n\nThường gặp, ADR >1/100\n\nToàn...",NaN,"Thông tin sản xuất\nBảo quản: Dưới 30°C, tránh...",NaN,https://www.pharmacity.vn/irbesartan-150mg-2-v...,"Thuốc tim mạch, huyết áp",Thuốc kê đơn,https://www.pharmacity.vn/rx-thuoc-tim-mach-hu...,"Thuốc tim mạch, huyết áp",NaN
2198,Viên nén Ihybes 150mg điều trị tăng huyết áp (...,NaN,['https://prod-cdn.pharmacity.io/e-com/images/...,Agimexpharm,NaN,NaN,Irbesartan,Điều trị tăng huyết áp,Viên nén bao phim,3 vỉ x 10 viên,...,Tác dụng phụ\nThường gặp (ADR > 1/100):\n\nThầ...,NaN,"Bảo quản\nNhiệt độ dưới 30oC, tránh ẩm và ánh ...",NaN,https://www.pharmacity.vn/ihybes-150-3-vi-x-10...,"Thuốc tim mạch, huyết áp",Thuốc kê đơn,https://www.pharmacity.vn/rx-thuoc-tim-mach-hu...,"Thuốc tim mạch, huyết áp",NaN
2199,Viên nén Dogrel Savi 75mg phòng ngừa thành lập...,NaN,['https://prod-cdn.pharmacity.io/e-com/images/...,Savi,NaN,NaN,Clopidogrel,"Phòng ngừa thành lập cục máu đông, huyết khối",Viên nén bao phim,3 vỉ x 10 viên,...,NaN,NaN,NaN,NaN,https://www.pharmacity.vn/dogrel-savi-75mg-3-v...,"Thuốc tim mạch, huyết áp",Thuốc kê đơn,https://www.pharmacity.vn/rx-thuoc-tim-mach-hu...,"Thuốc tim mạch, huyết áp",NaN


In [4]:
pharmcity_copy_df = pharmacity_df.copy()
pharmcity_copy_df.isnull().sum()

name                   482
price                 1169
images                 469
nha_san_xuat           548
noi_san_xuat          1884
thuoc_can_ke_don      1649
hoat_chat              632
chi_dinh_tom_tat       657
dang_bao_che           638
quy_cach               711
luu_y                  674
mo_ta                 1495
thanh_phan            1088
chi_dinh_chi_tiet     1365
huong_dan_su_dung     1092
cach_su_dung          2200
than_trong            1339
thong_tin_san_xuat    2200
url                      0
category               469
drug_type              469
category_link          469
category_name          469
error                 1731
dtype: int64

In [5]:
pharmcity_copy_df.iloc[1031:1250, 0:10]

,name,price,images,nha_san_xuat,noi_san_xuat,thuoc_can_ke_don,hoat_chat,chi_dinh_tom_tat,dang_bao_che,quy_cach
1031,Dung dịch uống Atiferlit 50mg/5ml An Thiên bổ ...,17.333 ₫/Ống,['https://prod-cdn.pharmacity.io/e-com/images/...,An Thien,NaN,Không,Sắt nguyên tố (dưới dạng Sắt (III) hydroxyd po...,Dung dịch thuốc uống Atiferlit giúp bổ sung th...,Dung dịch uống,Hộp 30 ống x 10ml
1032,Elnitine Stella (Hop 20 ong 10ml),80.000 ₫/Hộp,['https://prod-cdn.pharmacity.io/e-com/images/...,STELLA,NaN,Không,"Calci glyceroposphat, Magnesium gluconate.",Bổ sung calcium và magnesium trong một số trườ...,Dung dịch uống,Hộp 20 ống 10ml
1033,Viên nén sủi bọt CalciumBoston 500mg bổ sung c...,58.990 ₫/Hộp,['https://prod-cdn.pharmacity.io/e-com/images/...,Boston,NaN,Không,Calcium (different salts in combination),NaN,Viên nén sủi bọt,NaN
1034,Enat 400 (3 vỉ x 10 viên) Mã mới – Bổ sung vit...,45.000 ₫/Vỉ,['https://prod-cdn.pharmacity.io/e-com/images/...,Mega Lifesciences,NaN,Không,Vitamin E 400IU,"Ngăn ngừa lão hóa da, điều trị dự phòng thiếu ...",Viên nang,3 vỉ x 10 viên
1035,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1245,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1246,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1247,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
def get_product_information(driver, label):
    try:
        # Tìm tất cả p tags có class text-sm (chứa label)
        labels = driver.find_elements(By.CSS_SELECTOR, "p[class*='text-sm'][class*='leading-14']")
        
        for label_el in labels:
            label_text = label_el.text.strip()
            
            if label.lower() in label_text.lower():
                # Lấy parent div grid rồi tìm value trong sibling
                parent = label_el.find_element(By.XPATH, "./ancestor::div[contains(@class, 'grid')][1]")
                # Tìm div chứa text value (không phải label)
                value_els = parent.find_elements(By.CSS_SELECTOR, "div")
                
                for val in value_els:
                    val_text = val.text.strip()
                    # Lấy value khác label
                    if val_text and val_text != label_text:
                        return val_text
    except:
        return None

In [7]:
def get_section_detail(driver, section_id):
    try:
        # Tìm element theo id
        heading = driver.find_element(By.ID, section_id)
        
        # Lấy siblings trực tiếp từ heading
        siblings = heading.find_elements(By.XPATH, "./following-sibling::*")
        
        content = []
        for sib in siblings:
            tag = sib.tag_name.lower()
            
            # Dừng khi gặp heading tiếp theo (h2 hoặc h3)
            if tag in ["h2", "h3"]:
                break
            
            # Lấy text
            t = sib.get_attribute("textContent") or ""
            t = t.strip()
            if t:
                content.append(t)
        
        return "\n".join(content) if content else None
    
    except:
        None

In [8]:
def get_product_images(driver):
    images = []
    
    try:
        # Tìm tất cả img elements trong product detail
        img_elements = driver.find_elements(By.CSS_SELECTOR, "img[src*='production-cdn'], img[src*='pharmacity']")
        
        for img in img_elements:
            src = img.get_attribute("src")
            
            # Nếu src rỗng, thử lấy từ srcset hoặc data-src
            if not src:
                src = img.get_attribute("srcset") or img.get_attribute("data-src")
            
            if src:
                # Extract URL đầy đủ từ production-cdn.pharmacity.io
                match = re.search(r'(https://production-cdn\.pharmacity\.io/[^\s]+)', src)
                if match:
                    url = match.group(1).split()[0]  # Loại bỏ size info nếu có
                    if url not in images:
                        images.append(url)
                # Hoặc lấy trực tiếp nếu là URL hợp lệ
                elif src.startswith('http') and 'pharmacity' in src:
                    if src not in images:
                        images.append(src)
    
    except Exception as e:
        print(f"Error getting images: {e}")
    
    return images

In [9]:
def get_medicine_data(driver):
    try:
        WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.CSS_SELECTOR, "#pmc-v2 > div.bg-white.pt-\[--pt-main-pt\].lg\:pt-0 > div > div.bg-white > div:nth-child(2) > div.mb-4.grid.grid-cols-1.gap-6.lg\:grid-cols-2.lg\:items-start > div.grid.grid-cols-1.gap-4 > div > div > div.flex.flex-col.gap-4")))
    except:
        pass

    try:
        name = driver.find_element(By.XPATH, "//*[@id='pmc-v2']/div[4]/div/div[1]/div[2]/div[2]/div[2]/div/div/div[2]/div[2]/h2").text.strip()
        time.sleep(1)
    except:
        name = None
    
    try:
        price = driver.find_element(By.CSS_SELECTOR, "#pmc-v2 > div.bg-white.pt-\[--pt-main-pt\].lg\:pt-0 > div > div.bg-white > div:nth-child(2) > div.mb-4.grid.grid-cols-1.gap-6.lg\:grid-cols-2.lg\:items-start > div.grid.grid-cols-1.gap-4 > div > div > div.flex.flex-col.gap-4 > div.rounded-xs.bg-surface-canvas > div > div.flex.flex-row.items-center.gap-2 > h3").text.strip()
        time.sleep(1)
    except:
        price = None

    images = get_product_images(driver)
    time.sleep(1)

    nha_san_xuat = get_product_information(driver, "Nhà sản xuất:")
    noi_san_xuat = get_product_information(driver, "Nơi sản xuất:")
    thuoc_can_ke_don = get_product_information(driver, "Thuốc cần kê đơn:")
    hoat_chat = get_product_information(driver,"Hoạt chất:")
    chi_dinh_tom_tat = get_product_information(driver, "Chỉ định:")
    dang_bao_che = get_product_information(driver, "Dạng bào chế:")
    quy_cach = get_product_information(driver, "Quy cách:")
    luu_y = get_product_information(driver, "Lưu ý:")
    
    try:
        time.sleep(1)
        btn_section = driver.find_element(By.CSS_SELECTOR,"#pmc-v2 > div.bg-white.pt-\[--pt-main-pt\].lg\:pt-0 > div > div.bg-white > div:nth-child(2) > div.relative.left-1\/2.right-1\/2.-ml-\[50vw\].-mr-\[50vw\].w-screen.bg-surface-neutral-canvas > div > div:nth-child(1) > div > div.max-md\:pb-2.md\:h-auto.md\:max-h-\[inherit\] > div > div.relative.flex.items-center.justify-center")
        time.sleep(1)
        btn = btn_section.find_element(By.TAG_NAME, "button")
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
        time.sleep(1)
        btn.click()
        time.sleep(1)
    except Exception as e:
        print(f"Error clicking 'Xem thêm' button: {e}")


    mo_ta = get_section_detail(driver, "mo-ta")
    thanh_phan = get_section_detail(driver, "thanh-phan")
    chi_dinh_chi_tiet = get_section_detail(driver, "chi-dinh")
    huong_dan_su_dung = get_section_detail(driver, "huong-dan-su-dung")
    cach_su_dung = get_section_detail(driver, "cach-su-dung")
    than_trong = get_section_detail(driver, "than-trong")
    thong_tin_san_xuat = get_section_detail(driver, "thong-tin-san-xuat")

    return {
        "name": name,
        "price": price,
        "images": images,
        "nha_san_xuat": nha_san_xuat,
        "noi_san_xuat": noi_san_xuat,
        "thuoc_can_ke_don": thuoc_can_ke_don,
        "hoat_chat": hoat_chat,
        "chi_dinh_tom_tat": chi_dinh_tom_tat,
        "dang_bao_che": dang_bao_che,
        "quy_cach": quy_cach,
        "luu_y": luu_y,
        "mo_ta": mo_ta,
        "thanh_phan": thanh_phan,
        "chi_dinh_chi_tiet": chi_dinh_chi_tiet,
        "huong_dan_su_dung": huong_dan_su_dung,
        "cach_su_dung": cach_su_dung,
        "than_trong": than_trong,
        "thong_tin_san_xuat": thong_tin_san_xuat
    }

<>:3: SyntaxWarning: invalid escape sequence '\['
<>:14: SyntaxWarning: invalid escape sequence '\['
<>:33: SyntaxWarning: invalid escape sequence '\['
<>:3: SyntaxWarning: invalid escape sequence '\['
<>:14: SyntaxWarning: invalid escape sequence '\['
<>:33: SyntaxWarning: invalid escape sequence '\['
C:\Users\Admin\AppData\Local\Temp\ipykernel_29432\832258993.py:3: SyntaxWarning: invalid escape sequence '\['
  WebDriverWait(driver, 3).until(EC.presence_of_element_located((By.CSS_SELECTOR, "#pmc-v2 > div.bg-white.pt-\[--pt-main-pt\].lg\:pt-0 > div > div.bg-white > div:nth-child(2) > div.mb-4.grid.grid-cols-1.gap-6.lg\:grid-cols-2.lg\:items-start > div.grid.grid-cols-1.gap-4 > div > div > div.flex.flex-col.gap-4")))
C:\Users\Admin\AppData\Local\Temp\ipykernel_29432\832258993.py:14: SyntaxWarning: invalid escape sequence '\['
  price = driver.find_element(By.CSS_SELECTOR, "#pmc-v2 > div.bg-white.pt-\[--pt-main-pt\].lg\:pt-0 > div > div.bg-white > div:nth-child(2) > div.mb-4.grid.grid-co

In [10]:
#Tìm ra link cuối cùng đã crawl
try:
    existing_data = pd.read_csv("pharmacity_medicines_data_demo(4).csv")
    crawled_urls = set(existing_data['url'].dropna())
    print(f"Đã crawl: {len(crawled_urls)} link")
except:
    crawled_urls = set()
    print("Chưa có file output")

Đã crawl: 2200 link


In [11]:
#Tìm ra link chưa crawl
df = pd.read_csv("pharmacity_medicine_links.csv")
uncrawled_df = df[~df['links'].isin(crawled_urls)].reset_index(drop=True)
print(f"Còn {len(uncrawled_df)} link chưa crawl")

Còn 1368 link chưa crawl


In [ ]:
# Crawl tiếp những link chưa crawl
if len(uncrawled_df) > 0:
    driver = set_up_driver()
    results = list(existing_data.to_dict('records')) if len(crawled_urls) > 0 else []
    
    for i, row in uncrawled_df.iterrows():
        url = row['links']
        drug_type = row.get('drug_type', '')
        category_link = row.get('category_link', '')
        category = row.get('category_name', '')

        try:
            driver.get(url)
            time.sleep(3)
            data = get_medicine_data(driver)
            data['url'] = url
            data['category'] = category
            data['drug_type'] = drug_type
            data['category_link'] = category_link
            data['category_name'] = category
            results.append(data)
            print(f"[{i+1}/{len(uncrawled_df)}] crawled: {data.get('name', 'N/A')}")
            
        except Exception as e:
            results.append({'url': url, 'error': str(e)})
            print(f"[{i+1}/{len(uncrawled_df)}] error: {url}")
        
        if (i + 1) % 10 == 0:
            pd.DataFrame(results).to_csv("pharmacity_medicines_data_demo(5).csv", index=False, encoding='utf-8-sig')
            print(f"Saved checkpoint: {len(results)} total records")

    driver.quit()
    pd.DataFrame(results).to_csv("pharmacity_medicines_data_demo(5).csv", index=False, encoding='utf-8-sig')
    print(f"Done! Total: {len(results)} records")
else:
    print("Tất cả link đã crawl!")

[1/1368] crawled: Viên nén bao phim Savi prolol 5mg điều trị tăng huyết áp, đau thắt ngực, suy tim (3 vỉ x 10 viên)
[2/1368] crawled: Viên nén A.T Bisoprolol 2.5mg điều trị tăng huyết áp, đau thắt ngực, suy tim (10 vỉ x 10 viên)
[3/1368] crawled: Viên nén Uperio 200mg điều trị suy tim (4 vỉ x 7 viên)
[4/1368] crawled: Viên nén Zafular 200mg điều trị tăng cholesterol máu, giảm mỡ máu (5 vỉ x 10 viên)
[5/1368] crawled: Viên nén Losartan 25mg TV điều trị tăng huyết áp (3 vỉ x 10 viên)
[6/1368] crawled: Viên nén Peruzi 12.5mg điều trị tăng huyết áp, suy tim (3 vỉ x 10 viên)
[7/1368] crawled: Viên nén Zoamco 40mg điều trị tăng cholesterol máu, giảm mỡ máu (2 vỉ x 15 viên)
[8/1368] crawled: Viên nén Viacoram 7mg/5mg điều trị tăng huyết áp (chai 30 viên)
[9/1368] crawled: Viên nén Troysar AM Troikaa điều trị tăng huyết áp, đau thắt ngực (10 vỉ x 10 viên)
[10/1368] crawled: Viên nén Maxxprolol 2,5mg Ampharco điều trị tăng huyết áp, đau thắt ngực, suy tim (10 vỉ x 10 viên)
Saved checkpoint: 221